In [1]:
import json
import os
import pandas as pd

In [2]:
results_dir = '/data/rosa/work_in_progress/dictionary_learning_demo/._resid_5__data_rosa_work_in_progress_compositional_interpretability_outputs_shoe_simple_two_level_lr0.0005_epochs30_batch8_warmup100_pythia_cls_head_batch_top_k'
# '/data/rosa/work_in_progress/dictionary_learning_demo/SAEs_k_search_combined'

# '/data/rosa/work_in_progress/dictionary_learning_demo/._k_search__data_rosa_work_in_progress_compositional_interpretability_outputs_shoe_simple_two_level_lr0.0005_epochs30_batch8_warmup100_pythia_cls_head_batch_top_k'
# './SAEs_dev_shoe_simple_two_level_standard_lambda1e-1'
# './SAEs_dev_shoe_simple_two_level_batch_top_k_640'

In [4]:
results = {}

for submodule_dir in os.listdir(results_dir):
    for trainer_dir in os.listdir(os.path.join(results_dir, submodule_dir)):
        folder = os.path.join(submodule_dir, trainer_dir)
        run_dir = os.path.join(results_dir, folder)
        try:
            with open(os.path.join(run_dir, 'eval_results.json')) as f:
                metrics = json.load(f)
                config_file = os.path.join(run_dir, 'config.json')
                with open(config_file) as f:
                    config = json.load(f)
                    # l1_penalty = config['trainer']['l1_penalty']
                    k = config['trainer']['k']
                    # target_l0 = config['trainer']['target_l0']
                # results[f'{submodule_dir}_lambda{l1_penalty}'] = metrics
                results[f'{submodule_dir}_k{k}'] = metrics
                # results[f'{submodule_dir}_target_l0{target_l0}'] = metrics
        except FileNotFoundError:
            print(f'No eval_results.json in {run_dir}')

In [5]:
# sort by submodule name
results = dict(sorted(results.items()))

In [6]:
results

{'resid_out_5_k180': {'l2_loss': 3.473290368914604,
  'l1_loss': 304.9779243469238,
  'l0': 180.44875526428223,
  'frac_variance_explained': 0.9374578259885311,
  'cossim': 0.9977167956531048,
  'l2_ratio': 0.9977712668478489,
  'relative_reconstruction_bias': 1.0000463053584099,
  'loss_original': 8.95723569393158,
  'loss_reconstructed': 9.167105734348297,
  'loss_zero': 8.761781454086304,
  'frac_recovered': 2.7435932755470276,
  'frac_alive': 0.07794189453125,
  'hyperparameters': {'n_inputs': 200, 'context_length': 1024}},
 'resid_out_5_k210': {'l2_loss': 3.1290038973093033,
  'l1_loss': 349.3319149017334,
  'l0': 210.62409496307373,
  'frac_variance_explained': 0.9492384120821953,
  'cossim': 0.9981506168842316,
  'l2_ratio': 0.9980989433825016,
  'relative_reconstruction_bias': 0.9999459236860275,
  'loss_original': 8.95723569393158,
  'loss_reconstructed': 9.142985224723816,
  'loss_zero': 8.761781454086304,
  'frac_recovered': 2.453845967538655,
  'frac_alive': 0.062255859375,

In [7]:
# get a df where the columns are submodule names, frac_variance_explained, l1_loss, l0, frac_alive, and frac_recovered
df = pd.DataFrame(results).T
df = df[['frac_variance_explained', 'l1_loss', 'l0', 'frac_alive', 'frac_recovered', 'loss_original', 'loss_reconstructed']]

# add a column for the difference between the original and reconstructed loss
df['loss_diff'] = df['loss_original'] - df['loss_reconstructed']
df['loss_diff'] = df['loss_diff'].abs()
# drop the original and reconstructed loss columns
df = df.drop(columns=['loss_original', 'loss_reconstructed'])

# sort by submodule names
df = df.sort_index()

# turn fractions into percentages
df['frac_alive'] = 100*df['frac_alive']
df['frac_recovered'] = 100*df['frac_recovered']
df['frac_variance_explained'] = 100*df['frac_variance_explained']
df['l0'] = df['l0'].astype(int)
df['l1_loss'] = df['l1_loss'].astype(int)
df['frac_alive'] = df['frac_alive'].astype(int)
df['frac_recovered'] = df['frac_recovered'].astype(int)
df['frac_variance_explained'] = df['frac_variance_explained'].astype(int)
df = df.apply(pd.to_numeric, errors='coerce')
# keep the first two decimal places for loss_diff
df['loss_diff'] = df['loss_diff'].round(2)
# move the frac_recovery column and its values to the end
df = df[['frac_variance_explained', 'l1_loss', 'l0', 'frac_alive', 'loss_diff', 'frac_recovered']]

# add % sign to the values in the first column
df['frac_variance_explained'] = df['frac_variance_explained'].astype(str) + '%'
df['frac_alive'] = df['frac_alive'].astype(str) + '%'
df['frac_recovered'] = df['frac_recovered'].astype(str) + '%'

# rename the columns
df.columns = ['% Variance Explained', 'L1', 'L0', '% Alive', 'CE Diff', '% CE Recovered']

In [8]:
df

,% Variance Explained,L1,L0,% Alive,CE Diff,% CE Recovered
resid_out_5_k180,93%,304,180,7%,0.21,274%
resid_out_5_k210,94%,349,210,6%,0.19,245%
resid_out_5_k250,96%,407,250,5%,0.15,220%
resid_out_5_k300,97%,477,300,4%,0.11,187%
resid_out_5_k400,98%,623,400,3%,0.05,147%
resid_out_5_k500,99%,809,500,3%,0.00,100%
